# Fine-tune `intfloat/e5-mistral-7b-instruct` on Structural Similarity Labels

This notebook prepares and fine-tunes on your **10k story pairs** using full stories and structural similarity labels.

It supports two objectives (choose one in config):
- `infonce`: triplet-style contrastive objective with positives / hard negatives / easy negatives
- `contrastive_mse`: regression objective to match cosine similarity to normalized structural score

Data mapping is done from:
- pair scores: `data/Alignment/asq_random_10000_pair_structural_scores.json`
- full stories: `data/Alignment/asq_random_10000_story_pair_alignments.json`


In [1]:
import os
import sys
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType


/home/shayan/miniconda3/envs/story_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'src').exists() and (c / 'data').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.')

PROJECT_ROOT = find_project_root()
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /tank/scratch/shayan/Projects/NarrativeSimilarity


In [3]:
CONFIG = {
    # Data
    'scores_path': PROJECT_ROOT / 'data' / 'Alignment' / 'asq_random_10000_pair_structural_scores.json',
    'alignments_path': PROJECT_ROOT / 'data' / 'Alignment' / 'asq_random_10000_story_pair_alignments.json',
    'output_dir': PROJECT_ROOT / 'artifacts' / 'e5_mistral_structural_finetune',

    # Model
    'model_name': 'intfloat/e5-mistral-7b-instruct',
    'max_length': 1024,
    'use_bfloat16': True,
    'hf_cache_dir': Path('/scratch/shayan/hf_cache'),

    # Training objective: 'infonce' or 'contrastive_mse'
    'loss_type': 'infonce',

    # Optimization
    'seed': 42,
    'train_frac': 0.9,
    'batch_size': 2,
    'num_epochs': 10,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.03,
    'grad_accum_steps': 8,
    'max_grad_norm': 1.0,
    'temperature': 0.05,

    # Pair bucket sampling strategy
    # Quartile bins: 0=lowest sim, 1=low-mid, 2=high-mid, 3=highest sim
    'hard_negative_bin': 2,
    'positive_bin': 3,
    'easy_negative_bins': [0, 1],
    'max_triplets_per_anchor': 8,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],

    # Logging
    'eval_every_steps': 100,
}

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
print('Output dir:', CONFIG['output_dir'])


Output dir: /tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune


In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A40


## Load pair scores and map to full stories


In [5]:
with open(CONFIG['scores_path'], 'r', encoding='utf-8') as f:
    scores = json.load(f)
with open(CONFIG['alignments_path'], 'r', encoding='utf-8') as f:
    alignments = json.load(f)

scores_df = pd.DataFrame(scores)
align_df = pd.DataFrame([
    {
        'pair_id': x.get('pair_id'),
        'story_a_id': (x.get('story_a') or {}).get('id'),
        'story_b_id': (x.get('story_b') or {}).get('id'),
        'story_a_text': (x.get('story_a') or {}).get('narrative'),
        'story_b_text': (x.get('story_b') or {}).get('narrative'),
    }
    for x in alignments
])

pair_df = scores_df.merge(
    align_df[['pair_id', 'story_a_text', 'story_b_text']],
    on='pair_id',
    how='left',
)

pair_df = pair_df.dropna(subset=['story_a_text', 'story_b_text', 'pred_event_rating_mean_joint']).copy()
pair_df['structural_score'] = pair_df['pred_event_rating_mean_joint'].astype(float)

# Min-max normalize for contrastive regression target in [0, 1]
min_s, max_s = pair_df['structural_score'].min(), pair_df['structural_score'].max()
pair_df['score_norm'] = (pair_df['structural_score'] - min_s) / (max_s - min_s + 1e-12)

# Quartile bins for positive/negative sampling.
# If many scores are identical (e.g., lots of zeros), qcut can fail due to duplicate bin edges.
try:
    pair_df['sim_bin'] = pd.qcut(
        pair_df['score_norm'], q=4, labels=[0, 1, 2, 3], duplicates='raise'
    ).astype(int)
except ValueError:
    # Fallback: rank-based quartiles guarantee 4 bins while preserving score order.
    ranked = pair_df['score_norm'].rank(method='first', pct=True)
    pair_df['sim_bin'] = pd.qcut(ranked, q=4, labels=[0, 1, 2, 3]).astype(int)
    print('qcut fallback used: duplicate score edges detected; applied rank-based quartiles.')

print('pairs:', len(pair_df))
print('score range:', (float(min_s), float(max_s)))
print('bin counts:', pair_df['sim_bin'].value_counts().sort_index().to_string())
pair_df[['pair_id', 'score_norm', 'sim_bin']].head()

qcut fallback used: duplicate score edges detected; applied rank-based quartiles.
pairs: 10000
score range: (1.583, 2.5243661841)
bin counts: sim_bin
0    2500
1    2500
2    2500
3    2500


,pair_id,score_norm,sim_bin
0,1x53zg__4g4wtc,0.716079,3
1,9rprs0__3986p3,0.000000,0
2,5w3bgs__7au7vs,0.000000,0
3,3w07bi__6iaas5,0.000000,0
4,2schv9__3wqhqs,0.539322,3


## Train/validation split


In [6]:
pair_df = pair_df.sample(frac=1.0, random_state=CONFIG['seed']).reset_index(drop=True)
cut = int(len(pair_df) * CONFIG['train_frac'])
train_df = pair_df.iloc[:cut].copy()
val_df = pair_df.iloc[cut:].copy()

print('train pairs:', len(train_df))
print('val pairs  :', len(val_df))

train pairs: 9000
val pairs  : 1000


## Build datasets for each objective


In [7]:
def format_story_for_e5(text: str) -> str:
    # e5-style prefix
    return f"passage: {str(text).strip()}"


class PairRegressionDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.rows = []
        for _, r in df.iterrows():
            self.rows.append({
                'text_a': format_story_for_e5(r['story_a_text']),
                'text_b': format_story_for_e5(r['story_b_text']),
                'target': float(r['score_norm']),
            })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def build_triplets(df: pd.DataFrame, cfg: Dict) -> List[Dict]:
    # Build directed edges so each story can serve as anchor.
    directed = []
    for _, r in df.iterrows():
        bin_id = int(r['sim_bin'])
        score = float(r['score_norm'])
        a_id, b_id = str(r['story_a_id']), str(r['story_b_id'])
        a_text, b_text = r['story_a_text'], r['story_b_text']

        directed.append({'anchor_id': a_id, 'anchor_text': a_text, 'other_id': b_id, 'other_text': b_text, 'bin': bin_id, 'score': score})
        directed.append({'anchor_id': b_id, 'anchor_text': b_text, 'other_id': a_id, 'other_text': a_text, 'bin': bin_id, 'score': score})

    by_anchor: Dict[str, List[Dict]] = {}
    for row in directed:
        by_anchor.setdefault(row['anchor_id'], []).append(row)

    triplets = []
    rng = random.Random(cfg['seed'])

    for anchor_id, rows in by_anchor.items():
        pos = [x for x in rows if x['bin'] == cfg['positive_bin']]
        hard = [x for x in rows if x['bin'] == cfg['hard_negative_bin']]
        easy = [x for x in rows if x['bin'] in cfg['easy_negative_bins']]

        if len(pos) == 0 or len(hard) == 0 or len(easy) == 0:
            continue

        rng.shuffle(pos)
        max_k = min(len(pos), cfg['max_triplets_per_anchor'])

        for p in pos[:max_k]:
            h = rng.choice(hard)
            e = rng.choice(easy)
            triplets.append({
                'anchor': format_story_for_e5(p['anchor_text']),
                'positive': format_story_for_e5(p['other_text']),
                'hard_negative': format_story_for_e5(h['other_text']),
                'easy_negative': format_story_for_e5(e['other_text']),
            })

    return triplets


class TripletDataset(Dataset):
    def __init__(self, triplets: List[Dict]):
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        return self.triplets[idx]

In [8]:
if CONFIG['loss_type'] == 'contrastive_mse':
    train_dataset = PairRegressionDataset(train_df)
    val_dataset = PairRegressionDataset(val_df)
    print('Using contrastive_mse with pair dataset.')
    print('train size:', len(train_dataset), '| val size:', len(val_dataset))

elif CONFIG['loss_type'] == 'infonce':
    train_triplets = build_triplets(train_df, CONFIG)
    val_triplets = build_triplets(val_df, CONFIG)
    train_dataset = TripletDataset(train_triplets)
    val_dataset = TripletDataset(val_triplets)
    print('Using infonce with triplet dataset.')
    print('train triplets:', len(train_dataset), '| val triplets:', len(val_dataset))

else:
    raise ValueError("CONFIG['loss_type'] must be one of: 'infonce', 'contrastive_mse'")


Using infonce with triplet dataset.
train triplets: 1343 | val triplets: 1


## Tokenization, model, and LoRA setup


In [9]:
def collate_pair(batch):
    return {
        'text_a': [x['text_a'] for x in batch],
        'text_b': [x['text_b'] for x in batch],
        'target': torch.tensor([x['target'] for x in batch], dtype=torch.float32),
    }


def collate_triplet(batch):
    return {
        'anchor': [x['anchor'] for x in batch],
        'positive': [x['positive'] for x in batch],
        'hard_negative': [x['hard_negative'] for x in batch],
        'easy_negative': [x['easy_negative'] for x in batch],
    }


if CONFIG['loss_type'] == 'contrastive_mse':
    collate_fn = collate_pair
else:
    collate_fn = collate_triplet

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

print('train batches:', len(train_loader), '| val batches:', len(val_loader))

train batches: 672 | val batches: 1


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Force HF downloads/cache to scratch to avoid filling home directory.
os.environ['HF_HOME'] = str(CONFIG['hf_cache_dir'])
os.environ['HF_HUB_CACHE'] = str(CONFIG['hf_cache_dir'] / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CONFIG['hf_cache_dir'] / 'transformers')

cache_dir = str(CONFIG['hf_cache_dir'])
print('Using cache_dir:', cache_dir)

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], cache_dir=cache_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    CONFIG['model_name'],
    dtype=torch.bfloat16 if (torch.cuda.is_available() and CONFIG['use_bfloat16']) else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    cache_dir=cache_dir,
)

lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['target_modules'],
    bias='none',
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

Using cache_dir: /scratch/shayan/hf_cache


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.96s/it]


trainable params: 41,943,040 || all params: 7,152,603,136 || trainable%: 0.5864


## Embedding and loss functions


In [10]:
def encode_texts(texts: List[str]) -> Dict[str, torch.Tensor]:
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=CONFIG['max_length'],
        return_tensors='pt',
    )


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-6)
    return summed / counts


def embed(texts: List[str]) -> torch.Tensor:
    toks = encode_texts(texts)
    toks = {k: v.to(device) for k, v in toks.items()}
    out = model(**toks)
    emb = mean_pool(out.last_hidden_state, toks['attention_mask'])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb


def loss_contrastive_mse(batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    emb_a = embed(batch['text_a'])
    emb_b = embed(batch['text_b'])
    pred = F.cosine_similarity(emb_a, emb_b, dim=-1)
    # cosine is [-1, 1], map to [0, 1]
    pred_01 = (pred + 1.0) / 2.0
    target = batch['target'].to(device)
    return F.mse_loss(pred_01, target)


def loss_infonce(batch: Dict[str, List[str]]) -> torch.Tensor:
    anc = embed(batch['anchor'])
    pos = embed(batch['positive'])
    hneg = embed(batch['hard_negative'])
    eneg = embed(batch['easy_negative'])

    pos_sim = F.cosine_similarity(anc, pos, dim=-1)
    hneg_sim = F.cosine_similarity(anc, hneg, dim=-1)
    eneg_sim = F.cosine_similarity(anc, eneg, dim=-1)

    logits = torch.stack([pos_sim, hneg_sim, eneg_sim], dim=1) / CONFIG['temperature']
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, labels)


## Train + evaluate


In [ ]:
steps_per_epoch = max(1, math.ceil(len(train_loader) / CONFIG['grad_accum_steps']))
num_train_steps = steps_per_epoch * CONFIG['num_epochs']
num_warmup_steps = int(num_train_steps * CONFIG['warmup_ratio'])

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_train_steps)

print('num_train_steps:', num_train_steps, '| warmup:', num_warmup_steps)


In [ ]:
checkpoint_every_epochs = 2
checkpoint_dir = CONFIG['output_dir'] / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print('Checkpoint dir:', checkpoint_dir)
print('Checkpoint frequency (epochs):', checkpoint_every_epochs)


In [ ]:
def evaluate(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='eval', leave=False):
            if CONFIG['loss_type'] == 'contrastive_mse':
                loss = loss_contrastive_mse(batch)
            else:
                loss = loss_infonce(batch)
            losses.append(float(loss.item()))
    model.train()
    return float(np.mean(losses)) if losses else float('nan')


model.train()
global_step = 0
running = []

for epoch in range(CONFIG['num_epochs']):
    pbar = tqdm(train_loader, desc=f'train epoch {epoch+1}/{CONFIG["num_epochs"]}')
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar, start=1):
        if CONFIG['loss_type'] == 'contrastive_mse':
            loss = loss_contrastive_mse(batch)
        else:
            loss = loss_infonce(batch)

        loss = loss / CONFIG['grad_accum_steps']
        loss.backward()
        running.append(float(loss.item()) * CONFIG['grad_accum_steps'])

        if step % CONFIG['grad_accum_steps'] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            mean_loss = float(np.mean(running[-50:]))
            pbar.set_postfix({'loss': f'{mean_loss:.4f}', 'step': global_step})

            if global_step % CONFIG['eval_every_steps'] == 0:
                val_loss = evaluate(model, val_loader)
                print(f'\nstep={global_step} train_loss={mean_loss:.4f} val_loss={val_loss:.4f}')

    val_loss = evaluate(model, val_loader)
    print(f'Epoch {epoch+1} done | val_loss={val_loss:.4f}')

    if (epoch + 1) % checkpoint_every_epochs == 0:
        ckpt_path = checkpoint_dir / f'epoch_{epoch+1:02d}'
        ckpt_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        with open(ckpt_path / 'train_config.json', 'w', encoding='utf-8') as f:
            json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, f, indent=2)
        print(f'Saved checkpoint: {ckpt_path}')

## Save adapter, tokenizer, and config


In [ ]:
save_dir = CONFIG['output_dir'] / f"{CONFIG['model_name'].replace('/', '__')}_{CONFIG['loss_type']}"
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

with open(save_dir / 'train_config.json', 'w', encoding='utf-8') as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, f, indent=2)

print('Saved adapter+tokenizer to:', save_dir)


## Notes

- For `contrastive_mse`, all pairs are used with soft targets from `score_norm`.
- For `infonce`, positives are from top quartile (`sim_bin=3`), hard negatives from next quartile (`sim_bin=2`), and easy negatives from lower quartiles (`sim_bin=0,1`).
- If GPU memory is tight, reduce `max_length`, `batch_size`, and/or increase `grad_accum_steps`.
- For multi-GPU servers, run with `torchrun` / Accelerate if needed.


## Contrastive MSE Training

From this cell downward, we switch to fine-tuning with **contrastive MSE loss**.
This section trains on **story pairs** directly (no triplets), where the target is the normalized structural similarity score.


In [11]:
# Build pair-only datasets/loaders for contrastive MSE (no triplets).
CONFIG_MSE = dict(CONFIG)
CONFIG_MSE['loss_type'] = 'contrastive_mse'
CONFIG_MSE['num_epochs'] = 10
CONFIG_MSE['eval_every_steps'] = 100
CONFIG_MSE['batch_size'] = 8

print('MSE config:')
print({k: CONFIG_MSE[k] for k in ['loss_type', 'num_epochs', 'batch_size', 'grad_accum_steps', 'lr']})

mse_train_dataset = PairRegressionDataset(train_df)
mse_val_dataset = PairRegressionDataset(val_df)

mse_train_loader = DataLoader(
    mse_train_dataset,
    batch_size=CONFIG_MSE['batch_size'],
    shuffle=True,
    collate_fn=collate_pair,
)
mse_val_loader = DataLoader(
    mse_val_dataset,
    batch_size=CONFIG_MSE['batch_size'],
    shuffle=False,
    collate_fn=collate_pair,
)

print('mse train pairs:', len(mse_train_dataset), '| mse val pairs:', len(mse_val_dataset))
print('mse train batches:', len(mse_train_loader), '| mse val batches:', len(mse_val_loader))


MSE config:
{'loss_type': 'contrastive_mse', 'num_epochs': 10, 'batch_size': 8, 'grad_accum_steps': 8, 'lr': 0.0001}
mse train pairs: 9000 | mse val pairs: 1000
mse train batches: 1125 | mse val batches: 125


In [12]:
# Load a fresh model/tokenizer instance for MSE training.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Keep HF cache on scratch.
os.environ['HF_HOME'] = str(CONFIG_MSE['hf_cache_dir'])
os.environ['HF_HUB_CACHE'] = str(CONFIG_MSE['hf_cache_dir'] / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CONFIG_MSE['hf_cache_dir'] / 'transformers')
cache_dir = str(CONFIG_MSE['hf_cache_dir'])

mse_tokenizer = AutoTokenizer.from_pretrained(CONFIG_MSE['model_name'], cache_dir=cache_dir)
if mse_tokenizer.pad_token is None:
    mse_tokenizer.pad_token = mse_tokenizer.eos_token

mse_model = AutoModel.from_pretrained(
    CONFIG_MSE['model_name'],
    dtype=torch.bfloat16 if (torch.cuda.is_available() and CONFIG_MSE['use_bfloat16']) else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    cache_dir=cache_dir,
)

mse_lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG_MSE['lora_r'],
    lora_alpha=CONFIG_MSE['lora_alpha'],
    lora_dropout=CONFIG_MSE['lora_dropout'],
    target_modules=CONFIG_MSE['target_modules'],
    bias='none',
)

mse_model = get_peft_model(mse_model, mse_lora_cfg)
mse_model.print_trainable_parameters()

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.95s/it]


trainable params: 41,943,040 || all params: 7,152,603,136 || trainable%: 0.5864


In [13]:
# MSE-specific embedding + loss helpers.
def mse_encode_texts(texts: List[str]) -> Dict[str, torch.Tensor]:
    return mse_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=CONFIG_MSE['max_length'],
        return_tensors='pt',
    )


def mse_embed(texts: List[str]) -> torch.Tensor:
    toks = mse_encode_texts(texts)
    toks = {k: v.to(device) for k, v in toks.items()}
    out = mse_model(**toks)
    emb = mean_pool(out.last_hidden_state, toks['attention_mask'])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb


def mse_loss_fn(batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    emb_a = mse_embed(batch['text_a'])
    emb_b = mse_embed(batch['text_b'])

    pred = F.cosine_similarity(emb_a, emb_b, dim=-1)
    pred_01 = (pred + 1.0) / 2.0

    target = batch['target'].to(device=device, dtype=pred_01.dtype)

    return F.mse_loss(pred_01, target)


def mse_evaluate(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='mse eval', leave=False):
            loss = mse_loss_fn(batch)
            losses.append(float(loss.item()))
    model.train()
    return float(np.mean(losses)) if losses else float('nan')

In [14]:
# Train with contrastive MSE and checkpoint every 2 epochs.
# If checkpoints exist, resume from the latest one; otherwise train from scratch.
from peft import PeftModel

mse_steps_per_epoch = max(1, math.ceil(len(mse_train_loader) / CONFIG_MSE['grad_accum_steps']))
mse_num_train_steps = mse_steps_per_epoch * CONFIG_MSE['num_epochs']
mse_num_warmup_steps = int(mse_num_train_steps * CONFIG_MSE['warmup_ratio'])

mse_checkpoint_every_epochs = 2
mse_checkpoint_dir = CONFIG_MSE['output_dir'] / 'checkpoints_mse_loss'
mse_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Detect latest checkpoint (epoch_XX_mse_loss)
ckpt_candidates = []
for d in mse_checkpoint_dir.glob('epoch_*_mse_loss'):
    if d.is_dir():
        name = d.name
        try:
            epoch_num = int(name.split('_')[1])
            ckpt_candidates.append((epoch_num, d))
        except Exception:
            pass

resume_epoch = 0
latest_ckpt_path = None
if ckpt_candidates:
    ckpt_candidates.sort(key=lambda x: x[0])
    resume_epoch, latest_ckpt_path = ckpt_candidates[-1]

if latest_ckpt_path is not None:
    print(f'Resuming MSE from checkpoint: {latest_ckpt_path} (completed epoch={resume_epoch})')
    mse_model = PeftModel.from_pretrained(mse_model, str(latest_ckpt_path), is_trainable=True)
    try:
        mse_tokenizer = AutoTokenizer.from_pretrained(str(latest_ckpt_path))
        if mse_tokenizer.pad_token is None:
            mse_tokenizer.pad_token = mse_tokenizer.eos_token
    except Exception as e:
        print(f'Warning: could not load tokenizer from checkpoint, keeping existing tokenizer. Error: {e}')
else:
    print('No existing MSE checkpoint found. Training from scratch.')

mse_optimizer = torch.optim.AdamW(
    mse_model.parameters(),
    lr=CONFIG_MSE['lr'],
    weight_decay=CONFIG_MSE['weight_decay'],
)
mse_scheduler = get_linear_schedule_with_warmup(
    mse_optimizer,
    num_warmup_steps=mse_num_warmup_steps,
    num_training_steps=mse_num_train_steps,
)

# Compute global step offset from completed epochs and advance scheduler state.
mse_global_step = resume_epoch * mse_steps_per_epoch
if mse_global_step > 0:
    for _ in range(mse_global_step):
        mse_scheduler.step()

print('mse_num_train_steps:', mse_num_train_steps, '| mse_warmup:', mse_num_warmup_steps)
print('MSE checkpoint dir:', mse_checkpoint_dir)
print('Resume epoch:', resume_epoch, '| starting global_step:', mse_global_step)

if resume_epoch >= CONFIG_MSE['num_epochs']:
    print('All configured MSE epochs already completed. Nothing to train.')
else:
    mse_model.train()
    mse_running = []

    for epoch in range(resume_epoch, CONFIG_MSE['num_epochs']):
        pbar = tqdm(mse_train_loader, desc=f'mse train epoch {epoch+1}/{CONFIG_MSE["num_epochs"]}')
        mse_optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar, start=1):
            loss = mse_loss_fn(batch)
            loss = loss / CONFIG_MSE['grad_accum_steps']
            loss.backward()
            mse_running.append(float(loss.item()) * CONFIG_MSE['grad_accum_steps'])

            if step % CONFIG_MSE['grad_accum_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(mse_model.parameters(), CONFIG_MSE['max_grad_norm'])
                mse_optimizer.step()
                mse_scheduler.step()
                mse_optimizer.zero_grad(set_to_none=True)
                mse_global_step += 1

                mse_mean_loss = float(np.mean(mse_running[-50:]))
                pbar.set_postfix({'mse_loss': f'{mse_mean_loss:.4f}', 'step': mse_global_step})

                if mse_global_step % CONFIG_MSE['eval_every_steps'] == 0:
                    mse_val_loss = mse_evaluate(mse_model, mse_val_loader)
                    print(f'\nmse step={mse_global_step} train_loss={mse_mean_loss:.4f} val_loss={mse_val_loss:.4f}')

        mse_val_loss = mse_evaluate(mse_model, mse_val_loader)
        print(f'MSE epoch {epoch+1} done | val_loss={mse_val_loss:.4f}')

        if (epoch + 1) % mse_checkpoint_every_epochs == 0:
            ckpt_path = mse_checkpoint_dir / f'epoch_{epoch+1:02d}_mse_loss'
            ckpt_path.mkdir(parents=True, exist_ok=True)
            mse_model.save_pretrained(ckpt_path)
            mse_tokenizer.save_pretrained(ckpt_path)
            with open(ckpt_path / 'train_config.json', 'w', encoding='utf-8') as f:
                json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG_MSE.items()}, f, indent=2)
            print(f'Saved MSE checkpoint: {ckpt_path}')


Resuming MSE from checkpoint: /tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_08_mse_loss (completed epoch=8)


/home/shayan/miniconda3/envs/story_env/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


mse_num_train_steps: 1410 | mse_warmup: 42
MSE checkpoint dir: /tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss
Resume epoch: 8 | starting global_step: 1128


mse train epoch 9/10:  51%|█████     | 576/1125 [27:48<7:14:45, 47.51s/it, mse_loss=0.0640, step=1200]


mse step=1200 train_loss=0.0640 val_loss=0.2366


...


mse step=1300 train_loss=0.0359 val_loss=0.2448


mse train epoch 10/10:  94%|█████████▍| 1056/1125 [51:41<54:33, 47.44s/it, mse_loss=0.0503, step=1400]  


mse step=1400 train_loss=0.0503 val_loss=0.2418


mse train epoch 10/10: 100%|██████████| 1125/1125 [54:44<00:00,  2.92s/it, mse_loss=0.0555, step=1408]


MSE epoch 10 done | val_loss=0.2417
Saved MSE checkpoint: /tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_10_mse_loss


In [ ]:
# Save final MSE adapter/tokenizer to a folder containing "mse_loss" in its name.
mse_save_dir = CONFIG_MSE['output_dir'] / f"{CONFIG_MSE['model_name'].replace('/', '__')}_mse_loss"
mse_save_dir.mkdir(parents=True, exist_ok=True)

mse_model.save_pretrained(mse_save_dir)
mse_tokenizer.save_pretrained(mse_save_dir)

with open(mse_save_dir / 'train_config.json', 'w', encoding='utf-8') as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG_MSE.items()}, f, indent=2)

print('Saved final MSE adapter+tokenizer to:', mse_save_dir)
